In [ ]:
import gc
import json
import os
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import earthaccess
import numpy as np
import xarray as xr

In [ ]:
# Set your NASA Earthdata token as an environment variable before running.
# Get a token at: https://urs.earthdata.nasa.gov/user-tokens
# Never hard-code credentials here.
#
# In your shell:   export EARTHDATA_TOKEN="your_token_here"
# Or in .netrc:    machine urs.earthdata.nasa.gov login <user> password <pass>
#
# earthaccess will also pick up EARTHDATA_USERNAME / EARTHDATA_PASSWORD automatically.
token = os.environ.get("EARTHDATA_TOKEN")
if not token:
    raise EnvironmentError(
        "EARTHDATA_TOKEN is not set. "
        "Export it in your shell before running this notebook."
    )


In [ ]:
# SHORT_NAME = "MITgcm_LLC4320_Pre-SWOT_JPL_L4_CapeBasin_v1.0"
VERSION = "1.0"
PROVIDER = "POCLOUD"  # PO.DAAC Earthdata Cloud provider for cloud-hosted granules

SHORT_NAME = os.getenv("EARTHDATA_SHORT_NAME", "MITgcm_LLC4320_Pre-SWOT_JPL_L4_CapeBasin_v1.0")


In [ ]:
# If this folder exists and contains *.nc files, the script will process locally.
# Otherwise it will search + stream from Earthdata Cloud (S3 in-region preferred).
INPUT_DIR = Path("capebasindata")   # empty → triggers cloud streaming mode


# OUTPUT_BASE = Path("cape_processed_data")
OUTPUT_BASE = Path("cape_processed_data")


In [ ]:
# Optional: restrict to time window to avoid pulling *all* granules
# e.g. TEMPORAL = ("2011-11-15", "2011-12-01")
TEMPORAL: Optional[Tuple[str, str]] = None

# Optional: limit number of granules/files processed (useful for testing)
LIMIT: Optional[int] = None

# Depth indices (2nd dimension) for U and V
U_SURF_K = 0
U_DEEP_K = 16
V_SURF_K = 0
V_DEEP_K = 16
THETA_K = 0

In [ ]:
# Storage optimization: cast arrays to float32 (set to None to keep native dtype)
OUTPUT_DTYPE = np.float32

# NetCDF engine for remote file-like objects (works well with fs.open(...))
REMOTE_XARRAY_ENGINE = "h5netcdf"

In [ ]:
# --------------------------------
# Helpers
# --------------------------------
def _maybe_cast(a: np.ndarray) -> np.ndarray:
    if OUTPUT_DTYPE is None:
        return a
    return a.astype(OUTPUT_DTYPE, copy=False)


def _get_da(ds: xr.Dataset, candidates: List[str]) -> xr.DataArray:
    for name in candidates:
        if name in ds:
            return ds[name]
    raise KeyError(f"None of these variables were found in the dataset: {candidates}. "
                   f"Available variables: {list(ds.data_vars)}")


def _extract_level_pos1(da: xr.DataArray, level_index: int) -> np.ndarray:
    """
    Extract level_index along the 2nd dimension (pos=1) regardless of dim name.
    Matches your original logic: da.values[:, level, :, :]
    """
    if da.ndim < 4:
        raise ValueError(f"Expected 4D array for {da.name}, got dims={da.dims}")
    level_dim = da.dims[1]
    return da.isel({level_dim: level_index}).to_numpy()


In [ ]:
def _ensure_output_dirs() -> None:
    var_dirs = [
        "eta", "us_s", "us_d", "vs_s", "vs_d", "theta", "dxc", "dyc", "oceaTAUX", "oceaTAUY"
    ]
    for v in var_dirs:
        (OUTPUT_BASE / v).mkdir(parents=True, exist_ok=True)


def _save_all(tag: str, arrays: Dict[str, np.ndarray]) -> None:
    np.save(OUTPUT_BASE / "eta" / f"{tag}.npy", arrays["eta"])
    np.save(OUTPUT_BASE / "us_s" / f"{tag}.npy", arrays["us_s"])
    np.save(OUTPUT_BASE / "us_d" / f"{tag}.npy", arrays["us_d"])
    np.save(OUTPUT_BASE / "vs_s" / f"{tag}.npy", arrays["vs_s"])
    np.save(OUTPUT_BASE / "vs_d" / f"{tag}.npy", arrays["vs_d"])
    np.save(OUTPUT_BASE / "theta" / f"{tag}.npy", arrays["theta"])
    np.save(OUTPUT_BASE / "dxc" / f"{tag}.npy", arrays["dxc"])
    np.save(OUTPUT_BASE / "dyc" / f"{tag}.npy", arrays["dyc"])
    np.save(OUTPUT_BASE / "oceaTAUX" / f"{tag}.npy", arrays["oceaTAUX"])
    np.save(OUTPUT_BASE / "oceaTAUY" / f"{tag}.npy", arrays["oceaTAUY"])


In [ ]:
def _extract_arrays(ds: xr.Dataset) -> Dict[str, np.ndarray]:
    eta_da = _get_da(ds, ["Eta", "eta"])
    U_da = _get_da(ds, ["U", "u"])
    V_da = _get_da(ds, ["V", "v"])
    theta_da = _get_da(ds, ["Theta", "theta"])
    dxc_da = _get_da(ds, ["DXC", "dxc"])
    dyc_da = _get_da(ds, ["DYC", "dyc"])
    taux_da = _get_da(ds, ["oceTAUX", "oceaTAUX"])
    tauy_da = _get_da(ds, ["oceTAUY", "oceaTAUY"])

    arrays = {
        "eta": _maybe_cast(eta_da.to_numpy()),
        "us_s": _maybe_cast(_extract_level_pos1(U_da, U_SURF_K)),
        "us_d": _maybe_cast(_extract_level_pos1(U_da, U_DEEP_K)),
        "vs_s": _maybe_cast(_extract_level_pos1(V_da, V_SURF_K)),
        "vs_d": _maybe_cast(_extract_level_pos1(V_da, V_DEEP_K)),
        "theta": _maybe_cast(_extract_level_pos1(theta_da, THETA_K)),
        "dxc": _maybe_cast(dxc_da.to_numpy()),
        "dyc": _maybe_cast(dyc_da.to_numpy()),
        "oceaTAUX": _maybe_cast(taux_da.to_numpy()),
        "oceaTAUY": _maybe_cast(tauy_da.to_numpy()),
    }
    return arrays


In [ ]:
def _dataset_time_bounds(ds: xr.Dataset) -> Tuple[Optional[str], Optional[str]]:
    if "time" in ds.coords:
        tvals = ds["time"].values
        if len(tvals) > 0:
            return str(tvals[0]), str(tvals[-1])
    return None, None


def _granule_time_bounds(granule: Any) -> Tuple[Optional[str], Optional[str]]:
    try:
        rdt = granule["umm"]["TemporalExtent"]["RangeDateTime"]
        return rdt.get("BeginningDateTime"), rdt.get("EndingDateTime")
    except Exception:
        return None, None



In [ ]:
def _sort_key_begin(granule: Any) -> str:
    t0, _ = _granule_time_bounds(granule)
    return t0 or ""


def _list_local_nc_files() -> List[Path]:
    if not INPUT_DIR.exists():
        return []
    return sorted(INPUT_DIR.glob("*.nc"))


In [ ]:
def _auth_and_search_granules() -> List[Any]:
    auth = earthaccess.login(strategy="netrc",persist=False)
    if not getattr(auth, "authenticated", False):
        raise RuntimeError(
            "Earthdata authentication failed. Ensure EARTHDATA_TOKEN (or "
            "EARTHDATA_USERNAME + EARTHDATA_PASSWORD) is set in the environment."
        )

    search_kwargs: Dict[str, Any] = dict(
        short_name=SHORT_NAME,
        provider=PROVIDER,
    )
    if TEMPORAL is not None:
        search_kwargs["temporal"] = TEMPORAL

    results = earthaccess.search_data(count=-1, **search_kwargs)
    results = sorted(results, key=_sort_key_begin)

    if LIMIT is not None:
        results = results[:LIMIT]

    return results


In [ ]:
def _download_test_granules(num_granules: int) -> int:
    """Download a small sample of granules to INPUT_DIR for quick validation runs."""
    if num_granules <= 0:
        return 0

    INPUT_DIR.mkdir(parents=True, exist_ok=True)
    results = _auth_and_search_granules()
    if not results:
        raise RuntimeError("No granules found to download for test mode.")

    sample = results[:num_granules]
    print(f"[INFO] Downloading {len(sample)} test granules to {INPUT_DIR} ...")
    downloaded = earthaccess.download(sample, local_path=str(INPUT_DIR))

    # Keep only NetCDF files for local processing.
    netcdf_count = sum(1 for p in downloaded if str(p).endswith((".nc", ".nc4")))
    print(f"[INFO] Download complete. NetCDF files downloaded: {netcdf_count}")
    return netcdf_count

In [ ]:
def _make_remote_fs(results: List[Any]) -> Tuple[Any, str, bool, str]:
    """
    Prefer S3 in-region filesystem (us-west-2). Fall back to HTTPS fsspec session.
    """
    try:
        fs = earthaccess.get_s3_filesystem(results=results)
        return fs, "direct", True, "s3"
    except Exception as e:
        print(f"[WARN] S3 direct filesystem unavailable ({e}). Falling back to HTTPS.")
        fs = earthaccess.get_fsspec_https_session()
        return fs, "external", False, "https"


def _pick_netcdf_link(granule: Any, access: str, in_region: bool) -> Optional[str]:
    links = granule.data_links(access=access, in_region=in_region)
    for u in links:
        if u.endswith(".nc") or u.endswith(".nc4"):
            return u
    return None


In [ ]:
def _refresh_s3_fs(results: List[Any]) -> Any:
    # Temporary credentials are time-limited; recreate fs if we hit auth/403 errors.
    return earthaccess.get_s3_filesystem(results=results)


In [ ]:
print(os.environ.get("EARTHDATA_TOKEN"))


In [ ]:
# Starting index for processing (set to None to process all, or an index to resume from)
# START_FROM_INDEX: Optional[int] = 381  # Set to None to start from beginning
START_FROM_INDEX: Optional[int] = None

# Process and archive in chunks so large jobs can be downloaded incrementally.
BATCH_SIZE = 100
ARCHIVE_PREFIX = "cape_processed_batch"
DELETE_BATCH_FILES_AFTER_ZIP = False  # Set True to free disk after each batch zip is created.

# Retry configuration
MAX_RETRIES = 3  # Maximum number of retries per granule
RETRY_DELAY = 5

In [ ]:
def _is_auth_error(exc: Exception) -> bool:
    """Check if exception is likely an authentication/authorization error."""
    exc_str = str(exc).lower()
    return any(
        keyword in exc_str
        for keyword in ["403", "forbidden", "unauthorized", "401", "credentials", "expired", "authentication"]
    )


In [ ]:
def main() -> None:
    import time
    import threading
    import zipfile
    from concurrent.futures import ThreadPoolExecutor, as_completed

    _ensure_output_dirs()
    manifest: List[Dict[str, Any]] = []

    var_dirs = [
        "eta", "us_s", "us_d", "vs_s", "vs_d", "theta", "dxc", "dyc", "oceaTAUX", "oceaTAUY"
    ]

    # if TEST_DOWNLOAD_GRANULES > 0:
    #     downloaded_count = _download_test_granules(TEST_DOWNLOAD_GRANULES)
    # if downloaded_count == 0:
    #     raise RuntimeError("Test download mode enabled, but no NetCDF files were downloaded.")
    # if TEST_DOWNLOAD_ONLY:
    #     print("[DONE] Test granule download finished. Set TEST_DOWNLOAD_ONLY=0 to process them.")
    #     return

    # -----------------------------
    # MODE 1: local files
    # -----------------------------
    local_files = _list_local_nc_files()
    if local_files:
        n = len(local_files)
        width = max(3, len(str(n)))
        print(f"[INFO] Found {n} local .nc files in {INPUT_DIR}. Processing locally.")

        for idx, path in enumerate(local_files):
            tag = f"{idx:0{width}d}"
            print(f"[{idx+1}/{n}] {path.name} -> {tag}.npy")

            with xr.open_dataset(path) as ds:
                arrays = _extract_arrays(ds)
                t0, t1 = _dataset_time_bounds(ds)

            _save_all(tag, arrays)

            manifest.append(
                {
                    "index": idx,
                    "tag": tag,
                    "mode": "local",
                    "source": str(path),
                    "time_start": t0,
                    "time_end": t1,
                }
            )

            del arrays
            gc.collect()

        (OUTPUT_BASE / "manifest.json").write_text(json.dumps(manifest, indent=2))
        print(f"[DONE] Wrote outputs under {OUTPUT_BASE} and manifest.json")
        return

    # -----------------------------
    # MODE 2: stream from Earthdata Cloud (S3 preferred)
    # -----------------------------
    print("[INFO] No local .nc files found; streaming from Earthdata Cloud via earthaccess.")
    results = _auth_and_search_granules()
    if not results:
        raise RuntimeError("No granules found for the configured search.")

    fs, access, in_region, mode = _make_remote_fs(results)

    granule_and_url: List[Tuple[Any, str]] = []
    for g in results:
        url = _pick_netcdf_link(g, access=access, in_region=in_region)
        if url is not None:
            granule_and_url.append((g, url))

    n = len(granule_and_url)
    if n == 0:
        raise RuntimeError("Found granules but could not resolve any NetCDF (.nc/.nc4) links.")

    # Load existing manifest to resume
    manifest_path = OUTPUT_BASE / "manifest.json"
    if manifest_path.exists():
        try:
            with open(manifest_path, "r") as f:
                existing_manifest = json.load(f)
            manifest = existing_manifest
            processed_indices = {item["index"] for item in manifest}
            print(f"[INFO] Found existing manifest with {len(processed_indices)} processed granules")
        except Exception as e:
            print(f"[WARN] Could not load existing manifest: {e}")
            processed_indices = set()
    else:
        processed_indices = set()

    # Apply START_FROM_INDEX filter
    start_idx = START_FROM_INDEX if START_FROM_INDEX is not None else 0

    # Filter granules to process
    granules_to_process = [
        (idx, g, url) for idx, (g, url) in enumerate(granule_and_url)
        if idx >= start_idx and idx not in processed_indices
    ]

    if not granules_to_process:
        print(f"[INFO] All granules already processed or no granules to process from index {start_idx}")
        (OUTPUT_BASE / "manifest.json").write_text(json.dumps(manifest, indent=2))
        print(f"[DONE] Wrote outputs under {OUTPUT_BASE} and manifest.json")
        return

    width = max(3, len(str(n)))
    batch_size = max(1, int(BATCH_SIZE))
    total_batches = (len(granules_to_process) + batch_size - 1) // batch_size

    print(
        f"[INFO] Processing {len(granules_to_process)} granules starting from index {start_idx} "
        f"in {total_batches} batch(es) of up to {batch_size}."
    )

    archive_dir = OUTPUT_BASE / "archives"
    archive_dir.mkdir(parents=True, exist_ok=True)

    def write_manifest() -> None:
        manifest_sorted = sorted(manifest, key=lambda x: x["index"])
        manifest_path.write_text(json.dumps(manifest_sorted, indent=2))

    def zip_batch_outputs(batch_results: List[Dict[str, Any]], batch_start_idx: int, batch_end_idx: int, batch_num: int) -> None:
        if not batch_results:
            print(
                f"[BATCH {batch_num}/{total_batches}] No successful outputs in "
                f"index range {batch_start_idx}-{batch_end_idx}; skipping zip."
            )
            return

        archive_name = f"{ARCHIVE_PREFIX}_{batch_start_idx:0{width}d}_{batch_end_idx:0{width}d}.zip"
        archive_path = archive_dir / archive_name

        batch_results_sorted = sorted(batch_results, key=lambda x: x["index"])
        tags = sorted({item["tag"] for item in batch_results_sorted})

        with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            for var_name in var_dirs:
                for tag in tags:
                    npy_path = OUTPUT_BASE / var_name / f"{tag}.npy"
                    if npy_path.exists():
                        zf.write(npy_path, arcname=f"{var_name}/{npy_path.name}")

            zf.writestr("manifest_batch.json", json.dumps(batch_results_sorted, indent=2))

        print(
            f"[BATCH {batch_num}/{total_batches}] Created archive {archive_path} "
            f"with {len(batch_results_sorted)} granule(s)."
        )

        if DELETE_BATCH_FILES_AFTER_ZIP:
            for var_name in var_dirs:
                for tag in tags:
                    npy_path = OUTPUT_BASE / var_name / f"{tag}.npy"
                    if npy_path.exists():
                        npy_path.unlink()
            print(f"[BATCH {batch_num}/{total_batches}] Removed batch .npy files after zipping.")

    # --- PARALLEL PROCESSING WITH RETRY LOGIC ---
    MAX_WORKERS = 1

    # Thread-safe lock for fs refresh
    fs_lock = threading.Lock()
    fs_container = {"fs": fs, "results": results}

    def refresh_filesystem() -> bool:
        """Thread-safe filesystem refresh."""
        with fs_lock:
            print("[INFO] Refreshing S3 credentials...")
            try:
                new_fs = _refresh_s3_fs(fs_container["results"])
                fs_container["fs"] = new_fs
                print("[INFO] S3 credentials refreshed successfully")
                return True
            except Exception as e:
                print(f"[ERROR] Failed to refresh filesystem: {e}")
                return False

    def process_one_granule(idx: int, g: Any, url: str) -> Optional[Dict[str, Any]]:
        tag = f"{idx:0{width}d}"
        granule_ur = g["umm"].get("GranuleUR", f"granule_{idx}")
        t0_meta, t1_meta = _granule_time_bounds(g)

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                print(f"[{idx+1}/{n}] Start {granule_ur} (attempt {attempt}/{MAX_RETRIES})")

                # Get current fs (thread-safe)
                with fs_lock:
                    current_fs = fs_container["fs"]

                with current_fs.open(url) as f:
                    with xr.open_dataset(f, engine=REMOTE_XARRAY_ENGINE) as ds:
                        arrays = _extract_arrays(ds)
                        t0_ds, t1_ds = _dataset_time_bounds(ds)

                _save_all(tag, arrays)

                print(f"[{idx+1}/{n}] Done  {granule_ur}")
                return {
                    "index": idx,
                    "tag": tag,
                    "mode": mode,
                    "granule_ur": granule_ur,
                    "source": url,
                    "time_start_meta": t0_meta,
                    "time_end_meta": t1_meta,
                    "time_start_ds": t0_ds,
                    "time_end_ds": t1_ds,
                    "attempts": attempt,
                }

            except Exception as exc:
                is_auth_error = _is_auth_error(exc)
                print(f"[{idx+1}/{n}] Attempt {attempt}/{MAX_RETRIES} failed for {granule_ur}: {exc}")

                # If auth error and not last attempt, try to refresh credentials
                if is_auth_error and attempt < MAX_RETRIES:
                    print(exc)
                    print(f"[{idx+1}/{n}] Detected auth error, attempting credential refresh...")
                    if refresh_filesystem():
                        print(f"[{idx+1}/{n}] Retrying after credential refresh...")
                        time.sleep(RETRY_DELAY)
                        continue

                # For non-auth errors or if refresh failed, just wait and retry
                if attempt < MAX_RETRIES:
                    print(f"[{idx+1}/{n}] Retrying in {RETRY_DELAY} seconds...")
                    time.sleep(RETRY_DELAY)
                else:
                    print(f"[{idx+1}/{n}] Failed permanently after {MAX_RETRIES} attempts: {granule_ur}")
                    return None

        return None

    for batch_num, batch_start in enumerate(range(0, len(granules_to_process), batch_size), start=1):
        batch_items = granules_to_process[batch_start:batch_start + batch_size]
        batch_start_idx = batch_items[0][0]
        batch_end_idx = batch_items[-1][0]

        print(
            f"[BATCH {batch_num}/{total_batches}] Processing granule indices "
            f"{batch_start_idx}-{batch_end_idx}"
        )

        batch_results: List[Dict[str, Any]] = []

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_idx = {
                executor.submit(process_one_granule, idx, g, url): idx
                for idx, g, url in batch_items
            }

            for future in as_completed(future_to_idx):
                res = future.result()
                if res:
                    manifest.append(res)
                    batch_results.append(res)
                    # Save manifest after each successful granule (fault tolerance + easy resume).
                    write_manifest()
                gc.collect()

        zip_batch_outputs(batch_results, batch_start_idx, batch_end_idx, batch_num)
        write_manifest()

    print(f"[DONE] Wrote outputs under {OUTPUT_BASE} and manifest.json")
    print(f"[DONE] Batch archives are available under {archive_dir}")


if __name__ == "__main__":
    main()

In [ ]:
import shutil
import os

def compress_processed_data(output_dir='cape_processed_data', archive_name='cape_processed_data'):
    """
    Compresses the processed data folder.
    Uses 'gztar' (tar.gz) which typically results in smaller file sizes than zip 
    for datasets with many files because it compresses the entire archive directory as one stream.

    Args:
        output_dir (str): The name of the directory to compress.
        archive_name (str): The name of the output archive (without extension).
    """
    if not os.path.exists(output_dir):
        print(f"Error: Directory '{output_dir}' does not exist.")
        return

    # 'gztar' produces a .tar.gz file
    output_filename = f"{archive_name}.tar.gz"
    print(f"Compressing '{output_dir}' into '{output_filename}'...")
    
    try:
        shutil.make_archive(archive_name, 'gztar', root_dir=output_dir)
        print(f"Successfully created '{output_filename}'. You can now download it.")
    except Exception as e:
        print(f"An error occurred while compressing: {e}")

# Run the compression function
compress_processed_data()